# Tutorial 2: Phase analysis with tree search
Dara is equipped with a parallelized tree search algorithm to identify possible phases
present in a given XRD pattern.

In this tutorial, we will try to identify the phases in one experimental solid-state
reaction sample between `GeO2` and `ZnO`.

> You can download this tutorial project from [here](https://CederGroupHub.github.io/dara/_static/tutorial.zip).

In [1]:
%pip install ipywidgets nbformat

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

from dara import search_phases

In [3]:
pattern_path = "tutorial_data/GeO2-ZnO_700C_60min.xrdml"

# three elements are present in the sample
chemical_system = "Ge-O-Zn"

## Step 1: Prepare reference phases

Dara pre-builds an index of all the unique and low-energy phases in ICSD and COD
databases. It also implements a method to download CIF structures from COD data server
so that there is no need to obtain the offline database.

Before every search, we will need to gather all the reference phases in the chemical
system for the search algorithm. Dara provides `ICSDDatabase` and `CODDatabase` to do
the filtering.

In this example, we will use `CODDatabase` to download all the phases in the chemical system of `Ge-O-Zn`.

In [4]:
import tempfile
import requests
from pathlib import Path
from dara.cif import Cif
from dara.structure_db import CODDatabase

# Windows fix: NamedTemporaryFile holds an exclusive lock on Windows, so Cif.from_file()
# cannot re-open it by name. Patch _download_cod to close the file before reading it.
def _download_cod_windows_fix(cod_id: str) -> Cif:
    COD_URL = "https://www.crystallography.net/cod/{cod_id}.cif"
    try:
        url = COD_URL.format(cod_id=cod_id)
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with tempfile.NamedTemporaryFile(mode="w+b", suffix=".cif", delete=False) as temp_file:
            temp_path = temp_file.name
            temp_file.write(response.content)
        # File is now closed — safe to re-open on Windows
        try:
            cif = Cif.from_file(temp_path)
        finally:
            Path(temp_path).unlink(missing_ok=True)
        cif.filename = str(cod_id)
    except Exception as e:
        print(f"Failed to download {cod_id}: {e}")
        raise
    return cif

CODDatabase._download_cod = staticmethod(_download_cod_windows_fix)

In [5]:
from dara.structure_db import CODDatabase

# The COD database contains methods to filter phases in the chemical system
cod_database = CODDatabase()

# gather reference phases and save them to a directory called "cifs"
all_icsd_ids = cod_database.get_cifs_by_chemsys(chemical_system, dest_dir="cifs")

2026-10-01 18:30:10,798 WARNING dara.structure_db Local copy of database not found. Attempting to download structures...


2026-10-01 18:30:38,265 INFO dara.structure_db Saving downloaded CIFs to dara_downloaded_cifs
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skipping common gas: O2
2026-10-01 18:30:38,365 INFO dara.structure_db Skippin

Since we are using a pre-filterd database (i.e., the COD), the downloaded CIF files will automatically be named according to the
following convention:

```
{composition}_{spacegroup}_(cod|icsd_{id})-{e_hull}.cif
```
Where the `e_hull` is the energy above the convex hull in meV/atom, as determined from
the Materials Project database for the ground-state entry with matching composition and spacegroup.

## Step 2: Search for phases

After preparing the reference CIFs, we can start the phase search on a provided XRD
pattern.

In this case, we are using the XRD pattern from the solid-state reaction sample
on our laboratory's Aeris diffractometer (`tutorial_data/GeO2-ZnO_700C_60min.xrdml`).

In [6]:
# gather all the phases in the "cifs" directory
all_cifs = list(Path("cifs").glob("*.cif"))

search_results = search_phases(
    pattern_path=pattern_path,
    phases=all_cifs,
    wavelength="Cu",
    instrument_profile="Aeris-fds-Pixcel1d-Medipix3",
)

2026-10-01 18:30:49,519	INFO worker.py:2024 -- Started a local Ray instance.


2026-10-01 18:31:01,007 INFO dara.search.tree rpb_threshold automatically set to 1.00 based on pattern SNR.
2026-10-01 18:31:01,012 INFO dara.search.tree Detecting peaks in the pattern.
2026-10-01 18:31:42,802 INFO dara.search.tree The wmax is automatically adjusted to 64.08.
2026-10-01 18:31:42,802 INFO dara.search.tree The intensity threshold is automatically set to 8.53 % of maximum peak intensity.
2026-10-01 18:31:42,802 INFO dara.search.tree Creating the root node.
2026-10-01 18:31:42,802 INFO dara.search.tree Refining all the phases in the dataset.
2026-10-01 18:37:36,867 INFO dara.search.tree The initial value of eps2 is automatically set to 0.000000_-0.05^0.05.
2026-10-01 18:37:36,867 INFO dara.search.tree Finished refining 23 phases, with 7 phases removed.
2026-10-01 18:37:36,867 INFO dara.search.tree Express mode is enabled. Grouping phases before starting.
2026-10-01 18:37:37,823 INFO dara.search.tree Phases are grouped into 15 groups. In express mode, only the best phase in

## Step 3: Result analysis
The returned search result will be a list of `SearchResult` object.

In [7]:
search_results

[SearchResult(refinement_result=RefinementResult(lst_data=LstResult(raw_lst='Rietveld refinement to file(s) GeO2-ZnO_700C_60min.xy\nBGMN version 4.2.22, 4986 measured points, 158 peaks, 25 parameters\nStart: Thu Oct  1 18:40:17 2026; End: Thu Oct  1 18:40:21 2026\n30 iteration steps\n\nRp=9.57%  Rpb=18.20%  R=10.11%  Rwp=11.95% Rexp=2.72%\nDurbin-Watson d=0.10\n1-rho=1.98%\n\nGlobal parameters and GOALs\n****************************\nQGeO2152cod23003650=0.4819+-0.0019\nQZnO186cod90041780=0.3837+-0.0021\nQZn2GeO4148cod90146310=0.1344+-0.0012\nEPS2=-0.002890+-0.000010\n\nLocal parameters and GOALs for phase GeO2152cod23003650\n******************************************************\nSpacegroupNo=152\nHermannMauguin=P3_121\nXrayDensity=4.276\nRphase=11.22%\nUNIT=NM\nA=0.499105+-0.000018\nC=0.564815+-0.000030\nk1=0.0100000\nB1=0.00500000\nGEWICHT=0.2650+-0.0010\nGrainSize(1,1,1)=84.1811\nAtomic positions for phase GeO2152cod23003650\n---------------------------------------------\n  3     0.

In this pattern, we only have one solution found with `Rwp = 12.04 %`.

In [8]:
for i in range(len(search_results)):
    print(f"Rwp of solution {i} = {search_results[i].refinement_result.lst_data.rwp} %")

Rwp of solution 0 = 11.95 %


Each `SearchResult` has a `.visualize()` method to visualize the refined pattern and
missing/extra peaks in the solution. If there are no missing or extra peaks, this option
will not appear.

In [9]:
search_results[0].visualize()

You can also view all the alternative phases in one solution from `SearchResult.phases` attribute.

In [10]:
print("Phases found in solution 0:")
for i, phases_ in enumerate(search_results[0].phases):
    print(f"    - Phase {i}: {[phase.path.name for phase in phases_]}")

Phases found in solution 0:
    - Phase 0: ['GeO2_154_(cod_9007477)-0.cif', 'GeO2_152_(cod_2300365)-0.cif']
    - Phase 1: ['ZnO_186_(cod_9004178)-0.cif']
    - Phase 2: ['Zn2GeO4_148_(cod_9014631)-0.cif']


From the result, you can see that for the phase `GeO2`, the algorithm identifies two
similar phases with slightly different spacegroups (152 and 154).